# Opening range shorts: SPY and two sector ETFs

This replays the **same 30-minute opening range, one-minute breakout-close signal, OR-line limit for 15 minutes, 1% stop, 2% target** across ticker days with an opening gap down greater than 1%. It keeps one pending order or open trade per ticker, exits by 3:55 ET, assumes 5 bps slippage on both sides, excludes November 4–5, and reports shorts filled before noon. Wins hit the full 2%; losses hit the stop or exit by time at/below −0.5%; other time exits are neutral.

The two largest groups in the user's sector mapping are **XLK (154 technology tickers)** and **XLV (85 health care tickers)**. Mapping is a snapshot of the 'Ticker groups' tab in 'Backtesting Tickers by Sector ETF'; it uses provisional sector proxies. Compare QQQ's opening gap within ±1% and negative move from 9:30 open immediately before entry with the *same* SPY rule, and with both rules together. For each group, also inspect the matching ETF's gap and direction at entry. Comparisons use the same trades with both QQQ and SPY data available; the notebook reports missing ETF history. Separate hypothetical filters are not additive positions. Repeat after excluding Nov 20, Oct 12/14/16, and then Nov 13 too.

In [ ]:
from google.colab import drive
from pathlib import Path
from IPython.display import display
import pandas as pd, numpy as np, re
drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/backtesting'),Path('/content/drive/MyDrive/Backtesting')] if p.is_dir()),None)
if ROOT is None: raise FileNotFoundError('Set ROOT to your Backtesting folder')
OUT=Path('/content/drive/MyDrive/Opening_Range_SPY_XLK_XLV');OUT.mkdir(exist_ok=True)
SLIPPAGE_BPS=5;WAIT_MIN=15
EXCLUDED={'2026-11-04','2026-11-05'}
STANDOUT={'2026-10-12','2026-10-14','2026-10-16'}
files={}
for p in ROOT.rglob('*_1m_*.csv'):
    m=re.match(r'(.+)_1m_\d{4}-\d{2}-\d{2}',p.stem)
    if m: files.setdefault(m.group(1).upper(),[]).append(p)
if 'QQQ' not in files:raise FileNotFoundError('QQQ 1-minute bars missing')
print('Ticker symbols with 1m files:',len(files),'| CSV files:',sum(map(len,files.values())))

def load_sessions(paths):
    groups={}
    for p in paths:
        try:
            a=pd.read_csv(p)
            col=next((c for c in a if c.lower() in ('datetime','date','timestamp')),a.columns[0])
            a.index=pd.to_datetime(a[col],utc=True,errors='coerce').dt.tz_convert('America/New_York')
            a=a[['Open','High','Low','Close','Volume']].apply(pd.to_numeric,errors='coerce').dropna(subset=['Open','High','Low','Close'])
            a=a.between_time('09:30','15:59')
            for date,g in a.groupby(a.index.date): groups.setdefault(str(date),[]).append(g)
        except Exception as e: print('Skipped',p.name,str(e)[:100])
    out={}
    for date,parts in groups.items():
        a=pd.concat(parts).sort_index();out[date]=a[~a.index.duplicated(keep='last')]
    return out

qqq=load_sessions(files['QQQ'])
market_dates=sorted(qqq)
prior_market={market_dates[j]:market_dates[j-1] for j in range(1,len(market_dates))}
def open_of(d,date):
    t=pd.Timestamp(date+' 09:30',tz='America/New_York')
    return float(d.loc[t,'Open']) if d is not None and t in d.index else np.nan
def close_of(d,date):
    if d is None or date is None:return np.nan
    t=pd.Timestamp(date+' 15:59',tz='America/New_York')
    return float(d.loc[t,'Close']) if d is not None and t in d.index else np.nan
def gap(op,prev):return 100*(op/prev-1) if np.isfinite(op) and np.isfinite(prev) and prev>0 else np.nan
qqq_gap={day:gap(open_of(qqq[day],day),close_of(qqq.get(prior_market.get(day)),prior_market.get(day))) for day in market_dates}
print('QQQ market dates:',len(market_dates))
SESSION_GAPS=Path('/content/drive/MyDrive/Opening_Range_All_Tickers_QQQ_At_Entry/all_ticker_session_gaps.csv')
if not SESSION_GAPS.exists():raise FileNotFoundError(SESSION_GAPS)
sessions=pd.read_csv(SESSION_GAPS,dtype={'ticker':str,'date':str})
chosen=sessions[sessions.ticker_gap_pct.lt(-1)&~sessions.date.isin(EXCLUDED)].copy()
needed=chosen.groupby('ticker').date.apply(set).to_dict()
print('Ticker gap-down days:',len(chosen),'| tickers:',len(needed))
if not needed:raise RuntimeError('No ticker-days pass saved opening gap filters')
markets={'QQQ':qqq}
for symbol in ('SPY','XLK','XLV'):
    markets[symbol]=load_sessions(files[symbol]) if symbol in files else {}
    print(symbol,'1-minute dates:',len(markets[symbol]))

In [ ]:
def breakout_events(d,date,minutes):
    start=pd.Timestamp(date+' 09:30',tz='America/New_York')
    opening=d.loc[(d.index>=start)&(d.index<start+pd.Timedelta(minutes=30))]
    hi=float(opening.High.max());lo=float(opening.Low.min())
    if minutes==1:bars=d
    else:
        bars=d.resample('2min',origin=start,label='left',closed='left').agg({'Open':'first','High':'max','Low':'min','Close':'last'})
        count=d.Close.resample('2min',origin=start,label='left',closed='left').count()
        bars=bars[count.eq(2)].dropna(subset=['Open','Close'])
    out=[];previous={'long':False,'short':False}
    for t,c in bars.iterrows():
        if t<start+pd.Timedelta(minutes=30):continue
        signal=t+pd.Timedelta(minutes=minutes)
        if signal.strftime('%H:%M')>'15:00':break
        up=bool(c.Close>hi);down=bool(c.Close<lo)
        if up and not previous['long']:out.append(dict(signal=signal,side='long',boundary=hi,or_high=hi,or_low=lo))
        if down and not previous['short']:out.append(dict(signal=signal,side='short',boundary=lo,or_high=hi,or_low=lo))
        previous={'long':up,'short':down}
    return out

def next_bar(d,signal):
    i=d.index.searchsorted(signal)
    return d.index[i] if i<len(d) and d.index[i]<=signal+pd.Timedelta(minutes=1) else None

def limit_entry(d,e):
    t=next_bar(d,e['signal'])
    if t is None:return None,'missing_next_open',e['signal']
    sign=1 if e['side']=='long' else -1
    price=e['boundary'];end=e['signal']+pd.Timedelta(minutes=WAIT_MIN)
    for q,c in d.loc[(d.index>=t)&(d.index<end)].iterrows():
        if sign*(float(c.Open)-price)<=0:
            return (q,float(c.Open),'open'),'filled',q
        if c.Low<=price if sign==1 else c.High>=price:
            return (q,price,'wick'),'filled',q
    return None,'unfilled_15m',end

def exit_trade(d,date,e,fill,target_pct,stop_pct):
    t,raw,fill_type=fill;sign=1 if e['side']=='long' else -1
    entry=raw*(1+sign*SLIPPAGE_BPS/10000)
    stop=entry*(1-sign*stop_pct);target=entry*(1+sign*target_pct)
    end=pd.Timestamp(date+' 15:55',tz='America/New_York')
    path=d.loc[(d.index>=t)&(d.index<=end)]
    if path.empty:return None
    exit_time=path.index[-1];exit_raw=float(path.iloc[-1].Close);outcome='time'
    for q,c in path.iterrows():
        bad=c.Low<=stop if sign==1 else c.High>=stop
        good=(c.High>=target if sign==1 else c.Low<=target) and not (q==t and fill_type=='wick')
        if bad or good:
            exit_time=q;exit_raw=stop if bad else target;outcome='stop' if bad else 'target';break
    exit_price=exit_raw*(1-sign*SLIPPAGE_BPS/10000)
    R=sign*(exit_price-entry)/entry/stop_pct
    result='win' if outcome=='target' else 'loss' if outcome=='stop' or R*stop_pct<=-.005 else 'neutral'
    return dict(entry_time=t,entry=entry,exit_time=exit_time,exit=exit_price,R=R,outcome=outcome,result=result,
                raw_fill_price=raw,fill_type=fill_type)

def price_group(price):
    if price<50:return 'Under $50'
    if price<100:return '$50–<$100'
    if price<200:return '$100–<$200'
    return '$200+'

In [ ]:
SECTOR_MAP=dict([["ACMR","XLK"],["ACN","XLK"],["ADI","XLK"],["AEHR","XLK"],["AKAM","XLK"],["ALAB","XLK"],["ALGM","XLK"],["AMAT","XLK"],["AMD","XLK"],["AMKR","XLK"],["ANET","XLK"],["APH","XLK"],["APLD","XLK"],["APP","XLK"],["APPN","XLK"],["ARM","XLK"],["ASX","XLK"],["AVGO","XLK"],["AVT","XLK"],["AXTI","XLK"],["BB","XLK"],["BILL","XLK"],["BRKR","XLK"],["CDNS","XLK"],["CDW","XLK"],["CGNX","XLK"],["CIEN","XLK"],["CLS","XLK"],["COHR","XLK"],["COHU","XLK"],["CRDO","XLK"],["CRM","XLK"],["CRWD","XLK"],["CRWV","XLK"],["DDOG","XLK"],["DELL","XLK"],["DOCN","XLK"],["DOX","XLK"],["DT","XLK"],["ENPH","XLK"],["ENTG","XLK"],["EPAM","XLK"],["ESTC","XLK"],["EXTR","XLK"],["FIG","XLK"],["FIVN","XLK"],["FLEX","XLK"],["FN","XLK"],["FORM","XLK"],["FOUR","XLK"],["FROG","XLK"],["FRSH","XLK"],["FSLR","XLK"],["FSLY","XLK"],["FTNT","XLK"],["GDDY","XLK"],["GFS","XLK"],["GLBE","XLK"],["GLW","XLK"],["GPN","XLK"],["GTLB","XLK"],["GWRE","XLK"],["HPE","XLK"],["HPQ","XLK"],["HUBS","XLK"],["INGM","XLK"],["INOD","XLK"],["INTC","XLK"],["INTU","XLK"],["IONQ","XLK"],["IOT","XLK"],["IT","XLK"],["JBL","XLK"],["KD","XLK"],["KEYS","XLK"],["KLAC","XLK"],["KVYO","XLK"],["LITE","XLK"],["LRCX","XLK"],["LSCC","XLK"],["MCHP","XLK"],["MDB","XLK"],["MGNI","XLK"],["MKSI","XLK"],["MNDY","XLK"],["MPWR","XLK"],["MRVL","XLK"],["MSTR","XLK"],["MTSI","XLK"],["MU","XLK"],["MXL","XLK"],["NBIS","XLK"],["NET","XLK"],["NOK","XLK"],["NOW","XLK"],["NTAP","XLK"],["NTNX","XLK"],["NTSK","XLK"],["NVDA","XLK"],["NVTS","XLK"],["NXPI","XLK"],["OKTA","XLK"],["ON","XLK"],["ONTO","XLK"],["ORCL","XLK"],["P","XLK"],["PANW","XLK"],["PATH","XLK"],["PEGA","XLK"],["PENG","XLK"],["PLTR","XLK"],["Q","XLK"],["QBTS","XLK"],["QCOM","XLK"],["QLYS","XLK"],["QUBT","XLK"],["RBRK","XLK"],["RGTI","XLK"],["RMBS","XLK"],["RNG","XLK"],["S","XLK"],["SAIL","XLK"],["SANM","XLK"],["SEDG","XLK"],["SMCI","XLK"],["SMTC","XLK"],["SNDK","XLK"],["SNOW","XLK"],["SOUN","XLK"],["STM","XLK"],["STX","XLK"],["SWKS","XLK"],["TDC","XLK"],["TEAM","XLK"],["TENB","XLK"],["TER","XLK"],["TSEM","XLK"],["TSM","XLK"],["TTAN","XLK"],["TTMI","XLK"],["TXN","XLK"],["U","XLK"],["UMC","XLK"],["VIAV","XLK"],["VICR","XLK"],["VSH","XLK"],["WDAY","XLK"],["WDC","XLK"],["WIX","XLK"],["WOLF","XLK"],["YOU","XLK"],["ZETA","XLK"],["ZM","XLK"],["ZS","XLK"],["ABVX","XLV"],["ACAD","XLV"],["ACHC","XLV"],["ADMA","XLV"],["ADPT","XLV"],["ALNY","XLV"],["ARQT","XLV"],["ARWR","XLV"],["ASND","XLV"],["AXGN","XLV"],["BEAM","XLV"],["BETA","XLV"],["BKD","XLV"],["BTSG","XLV"],["CELC","XLV"],["CGON","XLV"],["CNC","XLV"],["COGT","XLV"],["CORT","XLV"],["CRSP","XLV"],["CYTK","XLV"],["DNTH","XLV"],["DOCS","XLV"],["EQPT","XLV"],["ERAS","XLV"],["EWTX","XLV"],["FRVO","XLV"],["GH","XLV"],["GMED","XLV"],["GRAL","XLV"],["HALO","XLV"],["HIMS","XLV"],["HUM","XLV"],["IBRX","XLV"],["ILMN","XLV"],["IMVT","XLV"],["INCY","XLV"],["INSM","XLV"],["IONS","XLV"],["IOVA","XLV"],["IRTC","XLV"],["LEGN","XLV"],["LFST","XLV"],["LGN","XLV"],["LIFE","XLV"],["LQDA","XLV"],["MMED","XLV"],["MRNA","XLV"],["NAVN","XLV"],["NTLA","XLV"],["NTRA","XLV"],["NVAX","XLV"],["NVST","XLV"],["ORKA","XLV"],["OSCR","XLV"],["PCVX","XLV"],["PODD","XLV"],["RARE","XLV"],["REPL","XLV"],["ROIV","XLV"],["RVMD","XLV"],["RVTY","XLV"],["RXRX","XLV"],["RYTM","XLV"],["SLS","XLV"],["SMMT","XLV"],["SRRK","XLV"],["SUPN","XLV"],["SYRE","XLV"],["TEM","XLV"],["TGTX","XLV"],["THC","XLV"],["TMDX","XLV"],["TNDM","XLV"],["TNGX","XLV"],["TVTX","XLV"],["TWST","XLV"],["TXG","XLV"],["VCEL","XLV"],["VCYT","XLV"],["VKTX","XLV"],["VRDN","XLV"],["WGS","XLV"],["XERS","XLV"],["XRAY","XLV"]])
rows=[];coverage=[]
for n,ticker in enumerate(sorted(needed),1):
    if ticker not in files:continue
    stock=load_sessions(files[ticker])
    for date in sorted(needed[ticker]):
        d=stock.get(date)
        if d is None:continue
        start=pd.Timestamp(date+' 09:30',tz='America/New_York')
        if not all(t in d.index for t in pd.date_range(start,periods=30,freq='min')):continue
        events=breakout_events(d,date,1)
        coverage.append(dict(ticker=ticker,date=date,bars=len(d),signals=len(events)))
        busy=start
        for j,e in enumerate(events,1):
            if e['signal']<=busy:continue
            fill,status,until=limit_entry(d,e)
            if fill is None:
                if status=='unfilled_15m':busy=until
                continue
            trade=exit_trade(d,date,e,fill,.02,.01)
            if trade is None:continue
            busy=trade['exit_time']+pd.Timedelta(minutes=1)
            rows.append(dict(ticker=ticker,date=date,sector=SECTOR_MAP.get(ticker,'OTHER'),
                signal_number=j,signal_time=e['signal'],side=e['side'],
                boundary=e['boundary'],price_group=price_group(e['boundary']),**trade))
    if n%100==0:print('Completed:',n,'/',len(needed))
    del stock
ledger=pd.DataFrame(rows)
if ledger.empty:raise RuntimeError('No fills in selected ticker gap-down days')
pd.DataFrame(coverage).to_csv(OUT/'ticker_date_coverage.csv',index=False)
ledger.to_csv(OUT/'all_replayed_fills.csv',index=False)
print('Ticker-days replayed:',len(coverage),'| all fills:',len(ledger))

In [ ]:
short=ledger[ledger.side.eq('short')].copy()
short['entry_et']=pd.to_datetime(short.entry_time,utc=True).dt.tz_convert('America/New_York')
short=short[short.entry_et.dt.hour.lt(12)].copy()

def market_gap(symbol,date):
    d=markets[symbol].get(date);prev=prior_market.get(date)
    return gap(open_of(d,date),close_of(markets[symbol].get(prev),prev))

def market_move(symbol,date,entry):
    d=markets[symbol].get(date)
    bar=entry-pd.Timedelta(minutes=1)
    op=open_of(d,date)
    return gap(float(d.loc[bar,'Close']),op) if d is not None and bar in d.index else np.nan

for symbol in ('QQQ','SPY','XLK','XLV'):
    short[symbol+'_gap_pct']=[market_gap(symbol,day) for day in short.date]
    short[symbol+'_move_pct']=[market_move(symbol,day,t) for day,t in zip(short.date,short.entry_et)]
short['sector_gap_pct']=np.where(short.sector.eq('XLK'),short.XLK_gap_pct,
                                  np.where(short.sector.eq('XLV'),short.XLV_gap_pct,np.nan))
short['sector_move_pct']=np.where(short.sector.eq('XLK'),short.XLK_move_pct,
                                   np.where(short.sector.eq('XLV'),short.XLV_move_pct,np.nan))
short.to_csv(OUT/'morning_shorts_market_context.csv',index=False)
print('Morning shorts after ticker gap-down >1%:',len(short))
for symbol in ('QQQ','SPY','XLK','XLV'):
    d=short[short.sector.eq(symbol)] if symbol in ('XLK','XLV') else short
    available=d[symbol+'_gap_pct'].notna()&d[symbol+'_move_pct'].notna()
    print(symbol,'context coverage:',int(available.sum()),'/',len(d),'trades; dates:',d.loc[available,'date'].nunique())
common=short[short.QQQ_gap_pct.notna()&short.QQQ_move_pct.notna()&
             short.SPY_gap_pct.notna()&short.SPY_move_pct.notna()].copy()
if common.empty:raise RuntimeError('No trade dates have both QQQ and SPY context; wait for SPY 1-minute backfill')
print('Matched-context morning trades:',len(common),'| dates:',common.date.nunique(),
      '| XLK tickers:',common.loc[common.sector.eq('XLK'),'ticker'].nunique(),
      '| XLV tickers:',common.loc[common.sector.eq('XLV'),'ticker'].nunique())
print('Matched dates:',sorted(common.date.unique()))

In [ ]:
def summarize(x,cohort,rule,period):
    w=int(x.result.eq('win').sum());l=int(x.result.eq('loss').sum());z=int(x.result.eq('neutral').sum())
    n=len(x);r=float(x.R.sum())
    return dict(period=period,cohort=cohort,rule=rule,trades=n,wins_2pct=w,losses=l,neutrals=z,
        win_rate_pct=round(100*w/n,1) if n else np.nan,net_R=round(r,3),
        mean_R=round(r/n,3) if n else np.nan,tickers=x.ticker.nunique(),dates=x.date.nunique())

EXCLUDE_FOUR={'2026-11-20','2026-10-12','2026-10-14','2026-10-16'}
EXCLUDE_FIVE=EXCLUDE_FOUR|{'2026-11-13'}
summary_rows=[];daily_rows=[]
for period,data in [('all_paired_dates',common),
                    ('without_four',common[~common.date.isin(EXCLUDE_FOUR)]),
                    ('without_five',common[~common.date.isin(EXCLUDE_FIVE)])]:
    for cohort in ('ALL','XLK','XLV'):
        group=data if cohort=='ALL' else data[data.sector.eq(cohort)]
        q=group.QQQ_gap_pct.abs().le(1)&group.QQQ_move_pct.lt(0)
        s=group.SPY_gap_pct.abs().le(1)&group.SPY_move_pct.lt(0)
        rules={'unfiltered':pd.Series(True,index=group.index),'QQQ':q,'SPY':s,'QQQ_and_SPY':q&s}
        if cohort!='ALL':
            etf=group.sector_gap_pct.abs().le(1)&group.sector_move_pct.lt(0)
            rules.update({'sector_ETF':etf,'QQQ_and_sector_ETF':q&etf,
                          'SPY_and_sector_ETF':s&etf})
        for name,mask in rules.items():
            selected=group[mask]
            summary_rows.append(summarize(selected,cohort,name,period))
            if period=='all_paired_dates':
                for date,day in selected.groupby('date'):
                    daily_rows.append(summarize(day,cohort,name,date))
report=pd.DataFrame(summary_rows)
report.to_csv(OUT/'filter_comparison.csv',index=False)
pd.DataFrame(daily_rows).to_csv(OUT/'filter_by_date.csv',index=False)
for period in report.period.unique():
    print('\n'+period);display(report[report.period.eq(period)])
print('Saved reports in',OUT)
print('Coverage caveat: ETF data gaps may concentrate the sector_ETF rows on recent dates.')